### Import modules

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager, rc
from pandas import Series, DataFrame
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, cross_validate, KFold, ShuffleSplit
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer, FunctionTransformer, LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss, mean_squared_error
from category_encoders import TargetEncoder, BinaryEncoder
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.distributions import CategoricalDistribution, IntDistribution, FloatDistribution
from optuna.integration import OptunaSearchCV, ShapleyImportanceEvaluator
from sklearn.cluster import KMeans
from tqdm import tqdm

### Step 1: Data Preprocessing

##### Read data

In [2]:
X_train = pd.read_csv('../data/X_train.csv', encoding='cp949')
y_train = pd.read_csv('../data/y_train.csv').gender
X_test = pd.read_csv('../data/X_test.csv', encoding='cp949')

# submission을 만들 때 사용하기 위해 ID 저정
id_test = X_test.id

##### Explore data

In [3]:
# 결측값 존재여부와 범주형 feature 확인
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 67 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  3500 non-null   int64  
 1   총구매액                3500 non-null   int64  
 2   구매건수                3500 non-null   int64  
 3   평균구매액               3500 non-null   int64  
 4   최대구매액               3500 non-null   int64  
 5   구매상품종류1             3500 non-null   int64  
 6   내점일수                3500 non-null   int64  
 7   구매주기                3500 non-null   int64  
 8   봄-구매비율              3500 non-null   float64
 9   여름-구매비율             3500 non-null   float64
 10  가을-구매비율             3500 non-null   float64
 11  겨울-구매비율             3500 non-null   float64
 12  환불금액                1205 non-null   float64
 13  환불건수                1205 non-null   float64
 14  내점당구매액              3500 non-null   float64
 15  내점당구매건수             3500 non-null   float64
 16  주구매상품 

In [5]:
# 결측값이 50퍼센트 이상인 칼럼 확인
missing_values = X_train.isnull().mean()
columns_to_drop = missing_values[missing_values >= 0.5].index.tolist()
columns_to_drop

['환불금액',
 '환불건수',
 '화장품구매주기',
 '주환불상품',
 '남성포함상품구매건수',
 '행사상품구매수',
 '만족도 떨어지는 제품 총 구입금액']

In [6]:
#결측값이 50퍼센트 이상인 칼럼 삭제
X_train.drop(columns = ['환불금액', '환불건수', '화장품구매주기', '주환불상품',
                        '남성포함상품구매건수', '행사상품구매수', '만족도 떨어지는 제품 총 구입금액'], axis =1)
X_test.drop(columns = ['환불금액', '환불건수', '화장품구매주기', '주환불상품',
                        '남성포함상품구매건수', '행사상품구매수', '만족도 떨어지는 제품 총 구입금액'], axis =1)

,id,총구매액,구매건수,평균구매액,최대구매액,구매상품종류1,내점일수,구매주기,봄-구매비율,여름-구매비율,...,의류_거래율(의류+식품+생활잡화),패션_대비_남성상품_거래율,패션_대비_여성상품_거래율,전체방문_중_환불방문(표준화),월_재방문율,시즌 마감 방문비율,상반기 구매비율,구매 추세,주구매지점재방문률,내점시마다 구매액의 일관성
0,3500,70900400,19,3731600,22000000,15,13,26,0.210526,0.052632,...,0.714286,0.461538,0.307692,0.356151,0.090909,0.307692,0.315789,0.368421,84.21,5.010000e+13
1,3501,310533100,219,1417959,38558000,37,90,3,0.091324,0.356164,...,0.500000,0.166667,0.083333,1.094810,0.818182,1.022222,0.141553,0.716895,99.54,5.630000e+13
2,3502,305264140,1477,206678,14825000,144,101,3,0.233582,0.283683,...,0.007951,0.000000,0.862069,0.056207,1.000000,5.485149,0.524712,-0.049425,99.93,1.510000e+13
3,3503,7594080,10,759408,5225000,7,5,47,0.100000,0.100000,...,0.142857,0.000000,0.500000,-0.485359,0.000000,1.000000,0.900000,-0.800000,90.00,4.460000e+12
4,3504,1795790,8,224474,1411200,6,3,8,0.000000,0.000000,...,0.000000,0.000000,0.000000,-0.485359,0.090909,2.000000,1.000000,-1.000000,87.50,3.320000e+11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2477,5977,82581500,14,5898679,23976000,8,8,40,0.071429,0.285714,...,0.800000,0.666667,0.111111,-0.485359,0.363636,0.000000,0.500000,0.000000,64.29,7.130000e+13
2478,5978,480000,1,480000,480000,1,1,0,0.000000,0.000000,...,1.000000,0.000000,0.000000,-0.485359,0.000000,1.000000,1.000000,-1.000000,0.00,0.000000e+00
2479,5979,260003790,71,3662025,25750000,38,19,18,0.154930,0.225352,...,0.516129,0.266667,0.400000,-0.485359,0.818182,1.421053,0.450704,0.098592,98.59,1.420000e+14
2480,5980,88991520,18,4943973,18120000,9,5,60,0.277778,0.166667,...,0.000000,0.000000,1.000000,-0.485359,0.000000,0.000000,0.444444,0.111111,94.44,5.820000e+13


In [6]:
# X_train에서 object 타입의 열(column) 확인하기
object_columns = X_train.select_dtypes(include=['object']).columns
print(object_columns)

Index(['주구매상품', '주구매지점', '고객등급', '주환불상품', '라이프 스타일(평균)', '라이프 스타일(건수)',
       '주구매 요일', '선호방문계절'],
      dtype='object')


In [7]:
# 범주형 변수와 수치형 변수를 분리
categorical_features = ['주구매상품', '주구매지점', '고객등급', '주환불상품', '라이프 스타일(평균)', '라이프 스타일(건수)', '주구매 요일', '선호방문계절']
numeric_features = list(set(X_train.columns)-set(categorical_features))

##### Impute missing values

In [8]:
from sklearn.impute import SimpleImputer 

if len(numeric_features) > 0:
    imp = SimpleImputer(strategy='mean')
    X_train[numeric_features] = imp.fit_transform(X_train[numeric_features])
    X_test[numeric_features] = imp.transform(X_test[numeric_features])
if len(categorical_features) > 0:  
    imp = SimpleImputer(strategy="most_frequent")
    X_train[categorical_features] = imp.fit_transform(X_train[categorical_features])
    X_test[categorical_features] = imp.transform(X_test[categorical_features])

In [10]:
# for i in X_train[categorical_features]:
#     print(X_train[i].value_counts())

In [11]:
# X_train['주환불상품'].value_counts() #=> 카디날리티가 데이터가 2482개임에비해 너무높음 => 

##### Transform features (Feature Scaling)

In [12]:
# DNN 모델링에서는 StandardScaler을 주로 사용
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

##### Encode Categorical Variables

In [16]:
# encoding
for i in tqdm(categorical_features):
    le = LabelEncoder()
    le=le.fit(X_train[i])
    X_train[i]=le.transform(X_train[i])

    for label in np.unique(X_test[i]):
        if label not in le.classes_: 
            le.classes_ = np.append(le.classes_, label)
    X_test[i]=le.transform(X_test[i])

100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 259.21it/s]


In [25]:
# # 중요도 1이상인것들끼리 조합의 합을 만듦
# X_train['상위2개합'] = X_train['주환불상품_넥타이(특정)']+X_train['주구매지점_전주점']
# X_train['3등4등합'] = X_train['주환불상품_4대 B/D']+X_train['주환불상품_편집매장']
# X_train['5등6등합'] = X_train['주환불상품_N.B']+X_train['주환불상품_보석']
# X_train['7등8등합'] = X_train['주환불상품_숙녀고정행사']+X_train['주환불상품_스트리트']
# X_train['9등10등합'] = X_train['주구매상품_액세서리']+X_train['주구매상품_침구/수예']



# X_test['상위2개합'] = X_test['주환불상품_넥타이(특정)']+X_test['주구매지점_전주점']
# X_test['3등4등합'] = X_test['주환불상품_4대 B/D']+X_test['주환불상품_편집매장']
# X_test['5등6등합'] = X_test['주환불상품_N.B']+X_test['주환불상품_보석']
# X_test['7등8등합'] = X_test['주환불상품_숙녀고정행사']+X_test['주환불상품_스트리트']
# X_test['9등10등합'] = X_test['주구매상품_액세서리']+X_test['주구매상품_침구/수예']

In [26]:
# # 1등별 2등 groupby
# X_train=pd.merge(X_train, X_train.groupby('주환불상품_넥타이(특정)')['주구매지점_전주점'].sum().reset_index().rename(columns={'주구매지점_전주점':'주환불상품_넥타이(특정)별 주구매지점_전주점sum'}), on = '주환불상품_넥타이(특정)',how='left')
# X_test=pd.merge(X_test, X_test.groupby('주환불상품_넥타이(특정)')['주구매지점_전주점'].sum().reset_index().rename(columns={'주구매지점_전주점':'주환불상품_넥타이(특정)별 주구매지점_전주점sum'}), on = '주환불상품_넥타이(특정)',how='left')

<font color='tomato'><font color="#CC3D3D"><p>
### Step 2: Modeling
아래 코드는 절대 수정하지 말것!!! 수정하여 submission 제출시 무조건 0점 처리함!!!

# CatBoost

In [18]:
models_CB = cross_validate(CatBoostClassifier(iterations = 1000, depth = 6, verbose=True, random_state=0),
                        X_train, y_train, 
                        cv=5, 
                        scoring='roc_auc', 
                        return_estimator=True, n_jobs=-1)
oof_pred_CB = np.array([m.predict_proba(X_test)[:,1] for m in models_CB['estimator']]).mean(axis=0)

scores = models_CB['test_score']
print("\nCatBoost CV scores: ", scores)
print("CatBoost CV mean = %.2f" % scores.mean(), "with std = %.2f" % scores.std())

Learning rate set to 0.015991
0:	learn: 0.6898652	total: 57.4ms	remaining: 57.3s
1:	learn: 0.6863249	total: 64ms	remaining: 31.9s
2:	learn: 0.6829026	total: 69.6ms	remaining: 23.1s
3:	learn: 0.6793025	total: 74.8ms	remaining: 18.6s
4:	learn: 0.6758138	total: 78.5ms	remaining: 15.6s
5:	learn: 0.6726681	total: 84.8ms	remaining: 14s
6:	learn: 0.6691465	total: 88ms	remaining: 12.5s
7:	learn: 0.6663246	total: 93.1ms	remaining: 11.5s
8:	learn: 0.6634790	total: 96.8ms	remaining: 10.7s
9:	learn: 0.6609867	total: 112ms	remaining: 11.1s
10:	learn: 0.6589149	total: 122ms	remaining: 11s
11:	learn: 0.6562531	total: 126ms	remaining: 10.3s
12:	learn: 0.6536582	total: 131ms	remaining: 9.91s
13:	learn: 0.6512835	total: 136ms	remaining: 9.56s
14:	learn: 0.6488342	total: 141ms	remaining: 9.28s
15:	learn: 0.6461816	total: 145ms	remaining: 8.91s
16:	learn: 0.6438766	total: 149ms	remaining: 8.61s
17:	learn: 0.6417446	total: 152ms	remaining: 8.31s
18:	learn: 0.6393825	total: 156ms	remaining: 8.03s
19:	learn:

In [21]:
sub1 = X_test[['id']]
sub1['gender'] = oof_pred_CB
sub1.to_csv('sub1.csv', index=False)

/tmp/ipykernel_674102/97427211.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub1['gender'] = oof_pred_CB


# ExtraTree

In [22]:
models_ET = cross_validate(ExtraTreesClassifier(n_estimators = 2000, random_state = 0, n_jobs=-1),
                        X_train, y_train, 
                        cv=5, 
                        scoring='roc_auc', 
                        return_estimator=True)
oof_pred_ET = np.array([m.predict_proba(X_test)[:,1] for m in models_ET['estimator']]).mean(axis=0)

scores = models_ET['test_score']
print("ET CV scores: ", scores)
print("ET CV mean = %.2f" % scores.mean(), "with std = %.2f" % scores.std())

ET CV scores:  [0.6949126  0.70619328 0.7696792  0.72957253 0.7526715 ]
ET CV mean = 0.73 with std = 0.03


criterion{“gini”, “entropy”, “log_loss”}, default=”gini” 바꿔가매해보세요

In [40]:
#pred_ET = model_ET.fit(X_train, y_train).predict_proba(X_test)[:,1]

In [23]:
sub2 = X_test[['id']]
sub2['gender'] = oof_pred_ET
sub2.to_csv('sub2.csv', index=False)

/tmp/ipykernel_674102/1998472725.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub2['gender'] = oof_pred_ET


# LGBM

In [24]:
# model_LGBM = LGBMClassifier(learning_rate = 0.01,num_leaves=31,n_estimators=1000, random_state = 0)

In [26]:
models_LGBM = cross_validate(LGBMClassifier(learning_rate = 0.1,n_estimators=1000,eval_metric='auc' , random_state = 0,n_jobs= -1),
                        X_train, y_train, 
                        cv=5, 
                        scoring='roc_auc', 
                        return_estimator=True)
oof_pred_LGBM = np.array([m.predict_proba(X_test)[:,1] for m in models_LGBM['estimator']]).mean(axis=0)

scores = models_LGBM['test_score']
print("LGBM CV scores: ", scores)
print("LGBM CV mean = %.2f" % scores.mean(), "with std = %.2f" % scores.std())

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Info] Number of positive: 1053, number of negative: 1747
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004991 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10207
[LightGBM] [Info] Number of data points in the train set: 2800, number of used features: 67
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.376071 -> initscore=-0.506257
[LightGBM] [Info] Start training from score -0.506257
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Info] Number of positive: 1053, number of negative: 1747
[LightGBM] [Info] A

In [38]:
#pred_LGBM = model_LGBM.fit(X_train, y_train).predict_proba(X_test)[:,1]

In [27]:
sub6 = X_test[['id']]
sub6['gender'] = oof_pred_LGBM
sub6.to_csv('sub6.csv', index=False)

/tmp/ipykernel_674102/2973373902.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub6['gender'] = oof_pred_LGBM


---

In [29]:
weighted_average = []
for i in range(len(oof_pred_CB)):  # 예측값들의 길이가 모두 같다고 가정
    weighted_average.append(oof_pred_CB[i] * 0.6 + oof_pred_ET[i] * 0.3 + oof_pred_LGBM[i] * 0.1)

In [30]:
final2 = X_test[['id']]
final2['gender'] = weighted_average
final2.to_csv('final2.csv', index=False)

/tmp/ipykernel_674102/1414716198.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final2['gender'] = weighted_average


<font color="#CC3D3D"><p>
# End